# 作业 02：根据三个线索，自己完成训练

请参考作业 01，独立编写完整的 PyTorch 训练流程，让下面的预测函数尽量拟合给定数据。

本题只给出三个线索。参数如何创建、如何计算梯度以及如何更新，都由你完成。可以查阅作业 01，但请根据本题的预测函数调整代码。

## 线索 1：预测函数

这次有三个待学习参数 $w_1,w_2,w_3$：

$$\hat y=w_1x+w_2x^2+w_3.$$

请根据这个公式，自己编写计算预测值的代码。输入整个 `x` 张量时，应得到每个样本对应的预测值。三个待学习参数也由你创建。请保留这个预测公式。

其中，$w_1$ 是一次项的系数，$w_2$ 是二次项的系数，$w_3$ 是常数项。三个参数在训练中都需要被调整。

## 线索 2：损失函数

每个样本的损失仍然是预测误差的绝对值：

$$L_i=|y_i-\hat y_i|.$$

本次有多个样本，请将所有样本的损失取平均，得到用于训练的一个数：

$$L=\frac{1}{N}\sum_{i=1}^{N}|y_i-\hat y_i|.$$

这里 $N$ 是样本数。请自己用 PyTorch 写出这个损失，不要换成平方误差。

## 线索 3：两个数据张量

`x[i]` 和 `y[i]` 是同一个样本的输入和目标。训练时保持这些数据不变，不要直接填写你猜测的参数作为答案。

In [2]:
import torch

# 输入数据与对应的目标数据，均为浮点数
x = torch.tensor([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
y = torch.tensor([-3.0, -1.375, 0.0, 1.125, 2.0, 2.625, 3.0, 3.125, 3.0])

## 你的作答区

请在下面编写完整训练代码，可以自行增加单元格。训练结束后，展示学到的三个参数、每个样本的预测值，以及最终的平均绝对误差。最终展示的预测值和损失应使用训练结束后的参数计算。

再用文字说明：你对作业 01 的训练代码做了哪些修改？你如何判断这次训练有效？不要求参数或误差恰好等于某个数，也不要求每一轮损失都下降。

In [7]:
w1 = torch.tensor(1.0, requires_grad=True)
w2 = torch.tensor(1.0, requires_grad=True)
w3 = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.Adam([w1, w2, w3], lr=0.01)

epochs = 5000
for epoch in range(epochs):
    y_hat = w1 * x + w2 * x ** 2 + w3

    loss = torch.abs(y - y_hat).mean()

    optimizer.zero_grad()   
    loss.backward()        
    optimizer.step()       

    if epoch % 1000 == 0:
        print(f"epoch {epoch:4d}  MAE={loss.item():.6f}  "
              f"w1={w1.item():.6f}  w2={w2.item():.6f}  w3={w3.item():.6f}")

with torch.no_grad():
    y_pred = w1 * x + w2 * x ** 2 + w3
    final_mae = torch.abs(y - y_pred).mean()

print("\n===== 最终结果 =====")
print(f"w1 = {w1.item():.6f}")
print(f"w2 = {w2.item():.6f}")
print(f"w3 = {w3.item():.6f}")
print("\n每个样本的预测值：")
for xi, yi, pi in zip(x, y, y_pred):
    print(f"  x={xi.item():+.1f}  目标 y={yi.item():+.3f}  预测 y_hat={pi.item():+.4f}")
print(f"\n最终平均绝对误差 MAE = {final_mae.item():.6f}")


epoch    0  MAE=1.888889  w1=1.010000  w2=0.990000  w3=-0.005984
epoch 1000  MAE=0.003933  w1=1.502172  w2=-0.498181  w3=2.002094
epoch 2000  MAE=0.003702  w1=1.497415  w2=-0.498237  w3=2.000964
epoch 3000  MAE=0.000544  w1=1.499969  w2=-0.499503  w3=2.000901
epoch 4000  MAE=0.001789  w1=1.499823  w2=-0.500917  w3=1.999729

===== 最终结果 =====
w1 = 1.499954
w2 = -0.501425
w3 = 1.999244

每个样本的预测值：
  x=-2.0  目标 y=-3.000  预测 y_hat=-3.0064
  x=-1.5  目标 y=-1.375  预测 y_hat=-1.3789
  x=-1.0  目标 y=+0.000  预测 y_hat=-0.0021
  x=-0.5  目标 y=+1.125  预测 y_hat=+1.1239
  x=+0.0  目标 y=+2.000  预测 y_hat=+1.9992
  x=+0.5  目标 y=+2.625  预测 y_hat=+2.6239
  x=+1.0  目标 y=+3.000  预测 y_hat=+2.9978
  x=+1.5  目标 y=+3.125  预测 y_hat=+3.1210
  x=+2.0  目标 y=+3.000  预测 y_hat=+2.9935

最终平均绝对误差 MAE = 0.003130


### 我的观察与说明


1. **参数**：从 1 个 `w` 变成 3 个 `w1、w2、w3`，都设置 `requires_grad=True`，并一起交给优化器更新。
2. **预测函数**：改为 ŷ = w1·x + w2·x² + w3，输入整个 x 张量时，逐样本输出对应的预测值。
3. **损失函数**：从平方误差改成平均绝对误差 `(y - y_hat).abs().mean()`，即 L = (1/N)Σ|yi − ŷi|，没有换成平方误差。
4. **优化器**：参数列表改为 `[w1, w2, w3]`。这里选用 Adam：L1 误差的梯度是误差的符号，批量 SGD 的符号梯度容易平均为 0、停在非最优点；Adam 的自适应步长能继续逼近最优点。


- 训练过程中 MAE 总体下降，并最终稳定在一个很小的数值；
- 最终每个样本的预测值和目标值接近；
- 三个参数收敛到稳定数值，不再大幅跳动；
- 不要求误差恰好等于 0，也不要求每一轮损失都下降，整体趋势向好即可。


## 提交

重启内核并从上到下运行全部单元格，确认代码可以完整运行。保存包含代码、输出和文字回答的本文件，再提交到自己的 GitHub 仓库。